## Part 2.1: Initialise Profiler Run State

### Objective

Initialise the shared runtime state required by the CSV schema diagnostic profiler before loading type rules, schema registries, or source files.

Each profiler execution receives one unique session ID. The same session ID is used throughout Parts 2.1–2.6 so that all audit events, file observations, schema records, and final summary information can be traced to one profiler run.

### Dependencies

This section depends on configuration and runtime objects established earlier in the notebook:

* `MELBOURNE_TIMEZONE` from Global Runtime Settings;
* `RAW_DATA_PATH` from Part 1.2;
* `LOG_AUDIT_PATH` from Part 1.2;
* `MASTER_SCHEMA_PATH` from Part 1.2;
* `PY_SCHEMA_PATH` from Part 1.2;
* `TYPE_RULES_PATH` from Part 1.2;
* `PIPELINE_AUDIT_PATH` from Part 1.2;
* `PROFILE_SAMPLE_ROWS` from Part 1.2.

This section does not reconstruct paths or reload project configuration.

### Run State

The profiler initialises:

* `SESSION_ID` to identify the current profiler execution;
* `PROFILER_START_TIME` using Melbourne local time;
* warning and error collections;
* counters for discovered, successful, failed, and structurally different files;
* collections for file-level observations, failures, and schema comparison details.

The session ID identifies an execution only. It is separate from the approved schema version.

### Central Audit Logging

A shared `log_event()` function is created for all profiler sections.

Each log entry contains:

* a timezone-aware Melbourne timestamp;
* the profiler session ID;
* a severity level;
* an event message.

Supported severity levels are:

* `INFO`
* `WARNING`
* `ERROR`
* `CRITICAL`

The function appends every event to `pipeline_audit.log`, prints the event in the notebook, and records warnings and errors in the current run state.

### Prerequisite Validation

Before the profiler begins, this section confirms that:

* all required path variables are available;
* `RAW_DATA_PATH` exists and is a directory;
* `LOG_AUDIT_PATH` exists and is a directory;
* `PIPELINE_AUDIT_PATH` exists and is a file;
* `PROFILE_SAMPLE_ROWS` contains a valid profiling policy.

Missing global prerequisites stop execution before any schema files or CSV files are processed.

### Expected Outcome

* A unique profiler session ID is generated.
* Melbourne profiler start time is recorded.
* Shared counters and result collections are initialised.
* Central audit logging is ready for Parts 2.2–2.6.
* Profiler prerequisites are verified.
* A `SESSION_START` event is appended to `pipeline_audit.log`.


In [ ]:
# ==========================================
# Part 2.1: Initialise Profiler Run State
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4


# ------------------------------------------
# Validate dependencies from earlier parts
# ------------------------------------------

REQUIRED_RUNTIME_VARIABLES = {
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "PROFILE_SAMPLE_ROWS": PROFILE_SAMPLE_ROWS,
}


for variable_name, variable_value in REQUIRED_RUNTIME_VARIABLES.items():
    if variable_value is None and variable_name != "PROFILE_SAMPLE_ROWS":
        raise RuntimeError(
            f"Required runtime variable is unavailable: {variable_name}"
        )


# ------------------------------------------
# Validate resolved path object types
# ------------------------------------------

REQUIRED_PATH_VARIABLES = {
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
}


for variable_name, path_value in REQUIRED_PATH_VARIABLES.items():
    if not isinstance(path_value, Path):
        raise TypeError(
            f"{variable_name} must be a pathlib.Path object."
        )


# ------------------------------------------
# Validate profiler prerequisites
# ------------------------------------------

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"RAW_DATA_PATH does not exist:\n{RAW_DATA_PATH}"
    )

if not RAW_DATA_PATH.is_dir():
    raise NotADirectoryError(
        f"RAW_DATA_PATH is not a directory:\n{RAW_DATA_PATH}"
    )


if not LOG_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"LOG_AUDIT_PATH does not exist:\n{LOG_AUDIT_PATH}"
    )

if not LOG_AUDIT_PATH.is_dir():
    raise NotADirectoryError(
        f"LOG_AUDIT_PATH is not a directory:\n{LOG_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"PIPELINE_AUDIT_PATH does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )

if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        f"PIPELINE_AUDIT_PATH is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if (
    PROFILE_SAMPLE_ROWS is not None
    and not isinstance(PROFILE_SAMPLE_ROWS, int)
):
    raise TypeError(
        "PROFILE_SAMPLE_ROWS must be an integer or null."
    )

if (
    isinstance(PROFILE_SAMPLE_ROWS, int)
    and PROFILE_SAMPLE_ROWS < 0
):
    raise ValueError(
        "PROFILE_SAMPLE_ROWS cannot be negative."
    )


# ------------------------------------------
# Initialise profiler execution identity
# ------------------------------------------

SESSION_ID = uuid4().hex[:8]

PROFILER_START_TIME = datetime.now(
    MELBOURNE_TIMEZONE
)


# ------------------------------------------
# Determine profiling policy
# ------------------------------------------

if PROFILE_SAMPLE_ROWS in (None, 0):
    PROFILING_MODE = "FULL_FILE"
    EFFECTIVE_SAMPLE_ROWS = None
else:
    PROFILING_MODE = "SAMPLED"
    EFFECTIVE_SAMPLE_ROWS = PROFILE_SAMPLE_ROWS


# ------------------------------------------
# Initialise run-level event collections
# ------------------------------------------

run_warnings = []
run_errors = []


# ------------------------------------------
# Initialise file-processing counters
# ------------------------------------------

files_discovered_count = 0
files_processed_successfully_count = 0
files_failed_count = 0
files_with_drift_count = 0
files_with_proposed_schema_variance_count = 0


# ------------------------------------------
# Initialise file-level result collections
# ------------------------------------------

discovered_files = []
processed_files = []
failed_files = []

observed_pandas_schemas_by_file = {}
observed_sql_schemas_by_file = {}

drift_details_by_file = {}
proposed_schema_variance_by_file = {}


# ------------------------------------------
# Define central profiler audit logger
# ------------------------------------------

SUPPORTED_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one profiler event to the pipeline audit log,
    print it in the notebook, and update run-level event
    collections when the event is a warning or error.
    """
    normalised_level = level.upper().strip()

    if normalised_level not in SUPPORTED_LOG_LEVELS:
        raise ValueError(
            f"Unsupported log level: {level}. "
            "Expected INFO, WARNING, ERROR, or CRITICAL."
        )

    if not isinstance(message, str) or not message.strip():
        raise ValueError(
            "Audit event message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).strftime("%Y-%m-%d %H:%M:%S %Z")

    log_entry = (
        f"[{timestamp}] "
        f"[RUN:{SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    # Append rather than overwrite so previous profiler
    # executions remain available for audit review.
    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(log_entry + "\n")

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "session_id": SESSION_ID,
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        run_warnings.append(event_record)

    elif normalised_level in {"ERROR", "CRITICAL"}:
        run_errors.append(event_record)

    return log_entry


# ------------------------------------------
# Record profiler session start
# ------------------------------------------

log_event(
    message=(
        "SESSION_START: CSV schema diagnostic profiler initialised. "
        f"Profiling mode={PROFILING_MODE}; "
        f"sample_rows={EFFECTIVE_SAMPLE_ROWS}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display initialisation summary
# ------------------------------------------

print("\n========== Profiler Run Initialisation ==========\n")
print(f"Session ID       : {SESSION_ID}")
print(
    "Start time       : "
    f"{PROFILER_START_TIME.isoformat(timespec='seconds')}"
)
print(f"Timezone         : {PROFILER_START_TIME.tzname()}")
print(f"Profiling mode   : {PROFILING_MODE}")
print(f"Sample rows      : {EFFECTIVE_SAMPLE_ROWS}")
print(f"Raw data path    : {RAW_DATA_PATH}")
print(f"Audit log path   : {PIPELINE_AUDIT_PATH}")

print("\n" + "=" * 55)
print("✓ Profiler run state initialised.")
print("✓ Runtime prerequisites verified.")
print("✓ Central audit logging is ready.")
print("=" * 55)

## Part 2.2: Create, Load, and Validate Type Rules

### Objective

Prepare the type rules used by the profiler to preserve important CSV columns, translate observed Pandas data types into SQL-compatible types, and apply controlled column-specific SQL overrides.

The rules are stored in `type_rules.json`, whose path was resolved earlier as `TYPE_RULES_PATH`.

### Type-Rule Structure

The rule file contains three sections:

```text
default_pandas_to_sql_mapping
column_name_to_sql_type_overrides
read_csv_explicit_dtypes
```

#### Pandas-to-SQL mappings

These rules translate observed Pandas data types into SQL-compatible definitions.

Examples include:

```text
int64           → INTEGER
float64         → NUMERIC(10, 6)
bool            → BOOLEAN
datetime64[ns]  → TIMESTAMP
object          → VARCHAR(255)
```

#### Column-specific SQL overrides

Some columns require explicit database definitions regardless of their inferred Pandas type.

For example, latitude and longitude fields use controlled numeric precision:

```text
start_lat → NUMERIC(10, 6)
start_lng → NUMERIC(10, 6)
end_lat   → NUMERIC(10, 6)
end_lng   → NUMERIC(10, 6)
```

Overrides use exact column-name matching. Broad substring matching is avoided because it could accidentally change unrelated columns.

#### Explicit CSV read dtypes

Known identifier and text columns are explicitly read as strings to reduce incorrect numeric inference and preserve values such as leading zeros.

Current explicit string columns include:

```text
start_station_name
start_station_id
end_station_name
end_station_id
```

### File-Management Policy

This section applies the following rules:

* If `type_rules.json` does not exist, create it once using the default rules.
* If the file already exists and is valid, preserve its existing values.
* If a required section is missing, add only the missing section from the defaults.
* If the JSON is invalid, preserve the existing file and use the in-memory defaults for the current run.
* If the top-level structure or a required section is malformed, preserve the file and use a safe in-memory replacement for the affected section.
* Never overwrite a valid existing rule file during normal execution.

When missing sections are added, the updated JSON is written atomically to reduce the risk of file corruption during an interrupted write.

### Safe Dtype Resolution

JSON stores dtype names as text, such as `"str"` or `"string"`.

The profiler converts these values through a controlled mapping rather than using `eval()`.

Supported dtype names include:

```text
str
object
int
float
bool
string
```

Unsupported dtype names generate a warning and use a safe `object` fallback for the current run.

### Expected Outcome

* `type_rules.json` exists or safe in-memory defaults are available.
* Existing valid rule values are preserved.
* Missing required sections are restored without replacing unrelated content.
* Explicit CSV dtypes are converted safely into Pandas-compatible dtype definitions.
* SQL mappings and column overrides are ready for later profiling stages.
* All repairs and fallbacks are recorded through the central profiler audit log.


In [ ]:
# ==========================================
# Part 2.2: Create, Load, and Validate
# Type Rules
# ==========================================

import json
from copy import deepcopy
from pathlib import Path

import pandas as pd


# ------------------------------------------
# Define default profiler type rules
# ------------------------------------------

DEFAULT_TYPE_RULES = {
    "default_pandas_to_sql_mapping": {
        "int64": "INTEGER",
        "Int64": "INTEGER",
        "float64": "NUMERIC(10, 6)",
        "bool": "BOOLEAN",
        "boolean": "BOOLEAN",
        "datetime64[ns]": "TIMESTAMP",
        "datetime64[ns, UTC]": "TIMESTAMP",
        "object": "VARCHAR(255)",
        "string": "VARCHAR(255)",
        "category": "VARCHAR(255)",
    },
    "column_name_to_sql_type_overrides": {
        "start_lat": "NUMERIC(10, 6)",
        "start_lng": "NUMERIC(10, 6)",
        "end_lat": "NUMERIC(10, 6)",
        "end_lng": "NUMERIC(10, 6)",
    },
    "read_csv_explicit_dtypes": {
        "start_station_name": "str",
        "start_station_id": "str",
        "end_station_name": "str",
        "end_station_id": "str",
    },
}


REQUIRED_TYPE_RULE_SECTIONS = {
    "default_pandas_to_sql_mapping",
    "column_name_to_sql_type_overrides",
    "read_csv_explicit_dtypes",
}


# ------------------------------------------
# Define controlled dtype-name mapping
# ------------------------------------------

SUPPORTED_DTYPE_NAMES = {
    "str": str,
    "object": object,
    "int": int,
    "float": float,
    "bool": bool,
    "string": pd.StringDtype(),
}


# ------------------------------------------
# Save JSON atomically
# ------------------------------------------

def save_json_atomically(
    file_path: Path,
    data: dict,
) -> None:
    """
    Write JSON to a temporary file and replace the target only
    after the write completes successfully.
    """
    temporary_path = file_path.with_suffix(
        file_path.suffix + ".tmp"
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as temporary_file:
            json.dump(
                data,
                temporary_file,
                indent=4,
                ensure_ascii=False,
            )

            temporary_file.write("\n")

        temporary_path.replace(file_path)

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


# ------------------------------------------
# Create default type rules if missing
# ------------------------------------------

def create_default_type_rules_if_missing(
    file_path: Path,
) -> bool:
    """
    Create type_rules.json only when it does not already exist.

    Returns True when the file is created and False when an
    existing file is preserved.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                "TYPE_RULES_PATH exists but is not a file:\n"
                f"{file_path}"
            )

        log_event(
            message=(
                "TYPE_RULES_FOUND: Existing type_rules.json "
                "will be preserved."
            ),
            level="INFO",
        )

        return False

    save_json_atomically(
        file_path=file_path,
        data=DEFAULT_TYPE_RULES,
    )

    log_event(
        message=(
            "TYPE_RULES_CREATED: type_rules.json was created "
            "using the default profiler rules."
        ),
        level="INFO",
    )

    return True


# ------------------------------------------
# Load and validate type rules
# ------------------------------------------

def load_type_rules(
    file_path: Path,
) -> dict:
    """
    Load type_rules.json conservatively.

    Invalid JSON or an invalid top-level structure is preserved
    on disk while the profiler uses in-memory defaults.
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as rule_file:
            loaded_rules = json.load(rule_file)

    except json.JSONDecodeError as error:
        log_event(
            message=(
                "TYPE_RULES_INVALID_JSON: "
                f"{file_path.name} could not be parsed. "
                "The existing file was preserved and in-memory "
                f"defaults will be used. Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    except OSError as error:
        log_event(
            message=(
                "TYPE_RULES_READ_FAILED: "
                f"{file_path.name} could not be read. "
                "In-memory defaults will be used. "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    if not isinstance(loaded_rules, dict):
        log_event(
            message=(
                "TYPE_RULES_INVALID_STRUCTURE: "
                f"{file_path.name} must contain a JSON object. "
                "The existing file was preserved and in-memory "
                "defaults will be used."
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_TYPE_RULES)

    validated_rules = deepcopy(loaded_rules)
    missing_sections = []
    invalid_sections = []

    for section_name in REQUIRED_TYPE_RULE_SECTIONS:

        if section_name not in validated_rules:
            validated_rules[section_name] = deepcopy(
                DEFAULT_TYPE_RULES[section_name]
            )

            missing_sections.append(section_name)

        elif not isinstance(
            validated_rules[section_name],
            dict,
        ):
            validated_rules[section_name] = deepcopy(
                DEFAULT_TYPE_RULES[section_name]
            )

            invalid_sections.append(section_name)

    # Missing sections represent incomplete but repairable JSON.
    # Add only those sections and preserve all existing content.
    if missing_sections:
        save_json_atomically(
            file_path=file_path,
            data=validated_rules,
        )

        log_event(
            message=(
                "TYPE_RULES_REPAIRED: Missing section(s) added "
                "from defaults: "
                + ", ".join(sorted(missing_sections))
            ),
            level="WARNING",
        )

    # Invalid section types are replaced only in memory.
    # The original file remains untouched for human review.
    if invalid_sections:
        log_event(
            message=(
                "TYPE_RULES_INVALID_SECTIONS: The following "
                "section(s) were not JSON objects and were "
                "replaced with defaults in memory only: "
                + ", ".join(sorted(invalid_sections))
            ),
            level="CRITICAL",
        )

    if not missing_sections and not invalid_sections:
        log_event(
            message=(
                "TYPE_RULES_LOADED: Existing type rules passed "
                "structural validation."
            ),
            level="INFO",
        )

    return validated_rules


# ------------------------------------------
# Resolve explicit CSV dtypes safely
# ------------------------------------------

def resolve_explicit_dtypes(
    configured_dtypes: dict,
) -> dict:
    """
    Convert configured dtype names into controlled Python or
    Pandas dtype definitions without using eval().
    """
    resolved_dtypes = {}

    for column_name, configured_dtype in configured_dtypes.items():

        if not isinstance(column_name, str) or not column_name.strip():
            log_event(
                message=(
                    "TYPE_RULE_INVALID_COLUMN: An explicit dtype "
                    "rule contained an invalid column name and was "
                    "ignored."
                ),
                level="WARNING",
            )

            continue

        if not isinstance(configured_dtype, str):
            log_event(
                message=(
                    "TYPE_RULE_INVALID_DTYPE: "
                    f"Column={column_name}; configured dtype must "
                    "be a string. Safe object fallback applied."
                ),
                level="WARNING",
            )

            resolved_dtypes[column_name] = object
            continue

        normalised_dtype = configured_dtype.strip()

        if normalised_dtype in SUPPORTED_DTYPE_NAMES:
            resolved_dtypes[column_name] = (
                SUPPORTED_DTYPE_NAMES[normalised_dtype]
            )

        else:
            log_event(
                message=(
                    "TYPE_RULE_UNSUPPORTED_DTYPE: "
                    f"Column={column_name}; "
                    f"dtype={configured_dtype}. "
                    "Safe object fallback applied."
                ),
                level="WARNING",
            )

            resolved_dtypes[column_name] = object

    return resolved_dtypes


# ------------------------------------------
# Initialise type rules
# ------------------------------------------

create_default_type_rules_if_missing(
    file_path=TYPE_RULES_PATH,
)

TYPE_RULES = load_type_rules(
    file_path=TYPE_RULES_PATH,
)


# ------------------------------------------
# Expose validated rule collections
# ------------------------------------------

PANDAS_TO_SQL_TYPE_MAP = TYPE_RULES[
    "default_pandas_to_sql_mapping"
]

COLUMN_SQL_TYPE_OVERRIDES = TYPE_RULES[
    "column_name_to_sql_type_overrides"
]

EXPLICIT_READ_DTYPES = resolve_explicit_dtypes(
    TYPE_RULES["read_csv_explicit_dtypes"]
)


# ------------------------------------------
# Display type-rule summary
# ------------------------------------------

print("\n========== Type Rule Initialisation ==========\n")

print(f"Type rules file       : {TYPE_RULES_PATH}")
print(
    "Pandas-to-SQL rules  : "
    f"{len(PANDAS_TO_SQL_TYPE_MAP)}"
)
print(
    "SQL column overrides : "
    f"{len(COLUMN_SQL_TYPE_OVERRIDES)}"
)
print(
    "Explicit CSV dtypes  : "
    f"{len(EXPLICIT_READ_DTYPES)}"
)

print("\nExplicit CSV dtype rules:")

for column_name, dtype_value in EXPLICIT_READ_DTYPES.items():
    print(f"  {column_name:<25}: {dtype_value}")

print("\n" + "=" * 55)
print("✓ Type rules created or loaded.")
print("✓ Required rule sections validated.")
print("✓ Explicit CSV dtypes resolved safely.")
print("=" * 55)